# 09 Beta Diversity and Diet

We test whether fungal community composition is associated with reported diet type (`diet_type_sample`). We compare whole-ASV and 16-mer Bray–Curtis distances using the same samples for both representations and both models. The adjusted model includes age, diet, and settlement type.

### Notebook overview

- **1. Setup**
    - 1.1 Imports
    - 1.2 Notebook settings and paths
    - 1.3 Load input<br><br>
    
    
- **2. Select samples**
    - 2.1 Match sample IDs
    - 2.2 Keep complete cases
    - 2.3 Summarize samples<br><br>
    
    
- **3. Overall PERMANOVA**
    - 3.1 Unadjusted PERMANOVA
        - 3.1.1 Whole ASVs
        - 3.1.2 16-mers
    - 3.2 Adjusted PERMANOVA
        - 3.2.1 Whole ASVs
        - 3.2.2 16-mers
    - 3.3 Compare unadjusted and adjusted results<br><br>
    
    
- **4. Pairwise PERMANOVA**
    - Prepare the pairwise inputs
    - 4.1 Unadjusted pairwise PERMANOVA
        - 4.1.1 Whole ASVs
        - 4.1.2 16-mers
    - 4.2 Adjusted pairwise PERMANOVA
        - 4.2.1 Whole ASVs
        - 4.2.2 16-mers
    - 4.3 Compare unadjusted and adjusted pairwise results<br><br>
    
    
- **5. Discussion of overall and pairwise PERMANOVA**

## 1. Setup

### 1.1 Imports

In [18]:
import subprocess
from pathlib import Path

import pandas as pd
import qiime2 as q2
from skbio import DistanceMatrix
from qiime2 import Visualization, Artifact, Metadata

from itertools import combinations # to create metadata combinations for pairwise PERMANOVA

### 1.2 Notebook settings and paths

We define input paths, shared analysis files, and separate outputs for each feature representation and model. The final meat-consumption section uses the original diet-labeled inputs.

In [19]:
# 1. Shared inputs and output directories
metadata_path = "../data/processed_ignored/06_diversity/Final_metadata_with_GHSL.tsv"
kmer16_distance_path = "../data/processed_ignored/07_beta_diversity/kmer16_bray_curtis_distance.qza"
asv_distance_path = "../data/processed_ignored/07_beta_diversity/asv_bray_curtis_distance.qza"

out_dir = "../data/processed_ignored/09_beta_diversity_diet"
vis_dir = "../data/processed/09_beta_diversity_diet"

columns = ["diet_type_sample", "age_years_sample", "Settlement_Type"]


# 2. Complete-case inputs for the main analysis
adjusted_diet_out_dir = f"{out_dir}/adjusted_diet"
adjusted_diet_vis_dir = f"{vis_dir}/adjusted_diet"

# Whole-ASV Bray–Curtis matrix
adjusted_distance_path = f"{adjusted_diet_out_dir}/asv_bray_curtis_complete.qza"

# 16-mer Bray–Curtis matrix
kmer16_complete_distance_path = f"{adjusted_diet_out_dir}/kmer16_bray_curtis_complete.qza"

# Shared metadata for both matrices
adjusted_metadata_path = f"{adjusted_diet_out_dir}/metadata_complete.tsv"

asv_complete_tsv_path = f"{adjusted_diet_out_dir}/asv_bray_curtis_complete.tsv"
kmer16_complete_tsv_path = f"{adjusted_diet_out_dir}/kmer16_bray_curtis_complete.tsv"


# 3. Overall PERMANOVA outputs
unadjusted_permanova_dir = f"{vis_dir}/unadjusted_permanova"
adjusted_permanova_dir = f"{vis_dir}/adjusted_permanova"

asv_unadjusted_overall_qzv = f"{unadjusted_permanova_dir}/asv_overall.qzv"
kmer16_unadjusted_overall_qzv = f"{unadjusted_permanova_dir}/kmer16_overall.qzv"

asv_unadjusted_overall_path = f"{unadjusted_permanova_dir}/asv_overall.tsv"
kmer16_unadjusted_overall_path = f"{unadjusted_permanova_dir}/kmer16_overall.tsv"

asv_adjusted_overall_qzv = f"{adjusted_permanova_dir}/asv_overall.qzv"
kmer16_adjusted_overall_qzv = f"{adjusted_permanova_dir}/kmer16_overall.qzv"

asv_adjusted_overall_path = f"{adjusted_permanova_dir}/asv_overall.tsv"
kmer16_adjusted_overall_path = f"{adjusted_permanova_dir}/kmer16_overall.tsv"


# 4. Pairwise PERMANOVA outputs
pairwise_input_dir = f"{adjusted_diet_out_dir}/pairwise_inputs"

asv_unadjusted_pairwise_path = f"{unadjusted_permanova_dir}/asv_pairwise.tsv"
kmer16_unadjusted_pairwise_path = f"{unadjusted_permanova_dir}/kmer16_pairwise.tsv"

asv_adjusted_pairwise_path = f"{adjusted_permanova_dir}/asv_pairwise.tsv"
kmer16_adjusted_pairwise_path = f"{adjusted_permanova_dir}/kmer16_pairwise.tsv"


# Paths for previous outputs
kmer16_permanova_path = f"{vis_dir}/kmer16_bray_curtis_diet_permanova.qzv"
asv_permanova_path = f"{vis_dir}/asv_bray_curtis_diet_permanova.qzv"
kmer16_pairwise_permanova_path = f"{vis_dir}/kmer16_bray_curtis_diet_pairwise_permanova.qzv"
asv_pairwise_permanova_path = f"{vis_dir}/asv_bray_curtis_diet_pairwise_permanova.qzv"
adjusted_permanova_path = f"{adjusted_diet_vis_dir}/asv_bray_curtis_adjusted_diet_permanova.qzv"

kmer16_aitchison_distance_path = "../data/processed_ignored/07_beta_diversity/kmer16_aitchison_distance.qza"
kmer16_aitchison_diet_distance_path = f"{out_dir}/kmer16_aitchison_diet.qza"
kmer16_aitchison_permanova_path = f"{vis_dir}/kmer16_aitchison_diet_permanova.qzv"
kmer16_aitchison_pairwise_permanova_path = f"{vis_dir}/kmer16_aitchison_diet_pairwise_permanova.qzv"


# Create directories
! mkdir -p "$out_dir" "$vis_dir" "$adjusted_diet_out_dir"
! mkdir -p "$unadjusted_permanova_dir" "$adjusted_permanova_dir"
! mkdir -p "$pairwise_input_dir"

### 1.3 Load inputs

We load the two Bray–Curtis matrices from notebook 07 and the cleaned metadata from notebook 06.

In [3]:
metadata = q2.Metadata.load(metadata_path).to_dataframe()
distance_matrix = q2.Artifact.load(kmer16_distance_path).view(DistanceMatrix)
asv_matrix = q2.Artifact.load(asv_distance_path).view(DistanceMatrix)

## 2. Select samples

### 2.1 Match sample IDs

We keep samples present in both Bray–Curtis matrices and in the metadata.

In [8]:
shared_ids = pd.Index(distance_matrix.ids).intersection(asv_matrix.ids)
shared_ids = shared_ids.intersection(metadata.index)

matched = metadata.loc[shared_ids, columns].copy()

diet = matched["diet_type_sample"]
matched["diet_type_sample"] = diet.str.strip().replace("", pd.NA)

print("Matched samples:", len(matched))

Matched samples: 150


### 2.2 Keep complete cases

We keep samples with diet, age, and settlement data. Both Bray–Curtis matrices are filtered to the same samples.

In [11]:
diet_samples = matched.dropna(subset=["diet_type_sample"])
complete = diet_samples.dropna()
complete_ids = complete.index.tolist()

distance = asv_matrix
distance_complete = distance.filter(complete_ids)
kmer16_complete = distance_matrix.filter(complete_ids)

assert list(distance_complete.ids) == complete_ids
assert list(kmer16_complete.ids) == complete_ids

Artifact.import_data("DistanceMatrix", distance_complete).save(adjusted_distance_path)
Artifact.import_data("DistanceMatrix", kmer16_complete).save(kmer16_complete_distance_path)

complete.index.name = "sample-id"
Metadata(complete).save(adjusted_metadata_path)
distance_complete.write(asv_complete_tsv_path)
kmer16_complete.write(kmer16_complete_tsv_path)

'../data/processed_ignored/09_beta_diversity_diet/adjusted_diet/kmer16_bray_curtis_complete.tsv'

### 2.3 Summarize samples

We compare diet group sizes before and after requiring complete metadata. We also check how the diet groups are distributed across settlement types.

In [25]:
counts = diet_samples["diet_type_sample"].value_counts().to_frame("With diet label")
counts["With all three variables"] = (
    complete["diet_type_sample"].value_counts().reindex(counts.index, fill_value=0)
)

display(counts)
display(pd.crosstab(complete["diet_type_sample"], complete["Settlement_Type"]))
print("Complete samples:", len(complete))

,With diet label,With all three variables
diet_type_sample,,
Omnivore,118,113
Vegetarian but eat seafood,8,8
Omnivore but do not eat red meat,8,7
Vegetarian,6,6
Vegan,5,5


Settlement_Type,Rural,Suburban,Urban
diet_type_sample,,,
Omnivore,49,20,44
Omnivore but do not eat red meat,2,2,3
Vegan,3,0,2
Vegetarian,4,1,1
Vegetarian but eat seafood,3,1,4


Complete samples: 139


### 3.1 Unadjusted PERMANOVA

We test whether fungal community composition differs across the five diet groups. Both tests use the same 139 samples and include diet as the only predictor.

#### 3.1.1 Whole ASVs

In [42]:
!qiime diversity adonis \
    --i-distance-matrix "$adjusted_distance_path" \
    --m-metadata-file "$adjusted_metadata_path" \
    --p-formula "diet_type_sample" \
    --o-visualization "$asv_unadjusted_overall_qzv"

Saved Visualization to: ../data/processed/09_beta_diversity_diet/unadjusted_permanova/asv_overall.qzv


In [ ]:
Visualization.load(asv_unadjusted_overall_qzv)

#### 3.1.2 16-mers

In [45]:
!qiime diversity adonis \
    --i-distance-matrix "$kmer16_complete_distance_path" \
    --m-metadata-file "$adjusted_metadata_path" \
    --p-formula "diet_type_sample" \
    --o-visualization "$kmer16_unadjusted_overall_qzv"

Saved Visualization to: ../data/processed/09_beta_diversity_diet/unadjusted_permanova/kmer16_overall.qzv


In [ ]:
Visualization.load(kmer16_unadjusted_overall_qzv)

#### Unadjusted results

| Diet term | Whole ASVs | 16-mers |
|---|---:|---:|
| Samples | 139 | 139 |
| R² | 0.0231 | 0.0187 |
| Pseudo-F | 0.791 | 0.639 |
| p-value | 0.868 | 0.960 |

Both unadjusted tests provided little evidence that fungal community composition differed among the five diet groups.

### 3.2 Adjusted PERMANOVA

We put diet last in the formula to test whether fungal community composition differs among the five diet groups after accounting for age and settlement type. Both tests use the same 139 samples as the unadjusted tests.

#### 3.2.1 Whole ASVs

In [46]:
!qiime diversity adonis \
    --i-distance-matrix "$adjusted_distance_path" \
    --m-metadata-file "$adjusted_metadata_path" \
    --p-formula "age_years_sample + Settlement_Type + diet_type_sample" \
    --o-visualization "$asv_adjusted_overall_qzv"

Saved Visualization to: ../data/processed/09_beta_diversity_diet/adjusted_diet/adjusted_permanova/asv_overall.qzv


In [ ]:
Visualization.load(asv_adjusted_overall_qzv)

#### 3.2.2 16-mers

In [47]:
!qiime diversity adonis \
    --i-distance-matrix "$kmer16_complete_distance_path" \
    --m-metadata-file "$adjusted_metadata_path" \
    --p-formula "age_years_sample + Settlement_Type + diet_type_sample" \
    --o-visualization "$kmer16_adjusted_overall_qzv"

Saved Visualization to: ../data/processed/09_beta_diversity_diet/adjusted_diet/adjusted_permanova/kmer16_overall.qzv


In [ ]:
Visualization.load(kmer16_adjusted_overall_qzv)

#### Adjusted results

| Diet term | Whole ASVs | 16-mers |
|---|---:|---:|
| Samples | 139 | 139 |
| R² | 0.0210 | 0.0165 |
| Pseudo-F | 0.730 | 0.573 |
| p-value | 0.949 | 0.987 |

After accounting for age and settlement type, both tests provided little evidence that fungal community composition differed among the five diet groups.

### 3.3 Compare unadjusted and adjusted PERMANOVA

All four tests used the same 139 samples. We compare the diet term before and after accounting for age and settlement type.

| Bray–Curtis representation | Model | Diet R² | Pseudo-F | p-value |
|---|---|---:|---:|---:|
| Whole ASVs | Unadjusted | 0.0231 | 0.791 | 0.868 |
| Whole ASVs | Adjusted | 0.0210 | 0.730 | 0.949 |
| 16-mers | Unadjusted | 0.0187 | 0.639 | 0.960 |
| 16-mers | Adjusted | 0.0165 | 0.573 | 0.987 |

The observed diet R² decreased slightly after adjustment for both representations. The observed diet R² was slightly higher for whole ASVs than for 16-mers in both models. All four tests provided little evidence of a diet association, so we can only interpret the observed differences in R² descriptively.

## 4. Pairwise PERMANOVA

We compare all ten pairs of diet groups on the whole-ASV and 16-mer Bray–Curtis matrices. For each pair, we use the same samples in the unadjusted and adjusted tests. We apply Holm correction across the ten comparisons separately for each representation and model.

### Prepare the pairwise inputs

We start from the 139 complete cases. For each diet pair, we save a matching metadata file and subset both distance matrices to those samples.

In [12]:
diet_pairs = list(combinations(sorted(complete["diet_type_sample"].unique()), 2))

for pair_number, (group_a, group_b) in enumerate(diet_pairs, start=1):
    pair_metadata = complete.loc[complete["diet_type_sample"].isin([group_a, group_b])]
    pair_ids = pair_metadata.index.tolist()

    Metadata(pair_metadata).save(f"{pairwise_input_dir}/pair_{pair_number}_metadata.tsv")

    asv_pair = distance_complete.filter(pair_ids)
    kmer16_pair = kmer16_complete.filter(pair_ids)

    Artifact.import_data("DistanceMatrix", asv_pair).save(
        f"{pairwise_input_dir}/pair_{pair_number}_asv.qza"
    )
    Artifact.import_data("DistanceMatrix", kmer16_pair).save(
        f"{pairwise_input_dir}/pair_{pair_number}_kmer16.qza"
    )

    print(f"{pair_number}. {group_a} vs. {group_b} (n = {len(pair_ids)})")

1. Omnivore vs. Omnivore but do not eat red meat (n = 120)
2. Omnivore vs. Vegan (n = 118)
3. Omnivore vs. Vegetarian (n = 119)
4. Omnivore vs. Vegetarian but eat seafood (n = 121)
5. Omnivore but do not eat red meat vs. Vegan (n = 12)
6. Omnivore but do not eat red meat vs. Vegetarian (n = 13)
7. Omnivore but do not eat red meat vs. Vegetarian but eat seafood (n = 15)
8. Vegan vs. Vegetarian (n = 11)
9. Vegan vs. Vegetarian but eat seafood (n = 13)
10. Vegetarian vs. Vegetarian but eat seafood (n = 14)


### 4.1 Unadjusted pairwise PERMANOVA

#### 4.1.1 Whole ASVs

We run one PERMANOVA per diet pair using the whole-ASV Bray–Curtis distances.

In [51]:
for pair_number, (group_a, group_b) in enumerate(diet_pairs, start=1):
    pair_distance_path = f"{pairwise_input_dir}/pair_{pair_number}_asv.qza"
    pair_metadata_path = f"{pairwise_input_dir}/pair_{pair_number}_metadata.tsv"
    pair_qzv_path = f"{unadjusted_permanova_dir}/asv_pair_{pair_number:02d}.qzv"

    print(f"{pair_number}. {group_a} vs. {group_b}")

    !qiime diversity adonis \
        --i-distance-matrix "$pair_distance_path" \
        --m-metadata-file "$pair_metadata_path" \
        --p-formula "diet_type_sample" \
        --o-visualization "$pair_qzv_path"

1. Omnivore vs. Omnivore but do not eat red meat
Saved Visualization to: ../data/processed/09_beta_diversity_diet/unadjusted_permanova/asv_pair_01.qzv
2. Omnivore vs. Vegan
Saved Visualization to: ../data/processed/09_beta_diversity_diet/unadjusted_permanova/asv_pair_02.qzv
3. Omnivore vs. Vegetarian
Saved Visualization to: ../data/processed/09_beta_diversity_diet/unadjusted_permanova/asv_pair_03.qzv
4. Omnivore vs. Vegetarian but eat seafood
Saved Visualization to: ../data/processed/09_beta_diversity_diet/unadjusted_permanova/asv_pair_04.qzv
5. Omnivore but do not eat red meat vs. Vegan
Saved Visualization to: ../data/processed/09_beta_diversity_diet/unadjusted_permanova/asv_pair_05.qzv
6. Omnivore but do not eat red meat vs. Vegetarian
Saved Visualization to: ../data/processed/09_beta_diversity_diet/unadjusted_permanova/asv_pair_06.qzv
7. Omnivore but do not eat red meat vs. Vegetarian but eat seafood
Saved Visualization to: ../data/processed/09_beta_diversity_diet/unadjusted_permano

In [1]:
# Visualization.load(f"{unadjusted_permanova_dir}/asv_pair_01.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/asv_pair_02.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/asv_pair_03.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/asv_pair_04.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/asv_pair_05.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/asv_pair_06.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/asv_pair_07.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/asv_pair_08.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/asv_pair_09.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/asv_pair_10.qzv")

#### Unadjusted pairwise results · Whole ASVs

| Diet pair | Samples | R² | Pseudo-F | p-value | Holm-adjusted p |
|---|---:|---:|---:|---:|---:|
| 1. Omnivore vs. Omnivore but do not eat red meat | 120 | 0.0045 | 0.534 | 0.950 | 1.000 |
| 2. Omnivore vs. Vegan | 118 | 0.0071 | 0.824 | 0.636 | 1.000 |
| 3. Omnivore vs. Vegetarian | 119 | 0.0085 | 1.001 | 0.400 | 1.000 |
| 4. Omnivore vs. Vegetarian but eat seafood | 121 | 0.0061 | 0.731 | 0.780 | 1.000 |
| 5. Omnivore but do not eat red meat vs. Vegan | 12 | 0.0875 | 0.959 | 0.446 | 1.000 |
| 6. Omnivore but do not eat red meat vs. Vegetarian | 13 | 0.0438 | 0.503 | 0.781 | 1.000 |
| 7. Omnivore but do not eat red meat vs. Vegetarian but eat seafood | 15 | 0.0620 | 0.859 | 0.596 | 1.000 |
| 8. Vegan vs. Vegetarian | 11 | 0.1308 | 1.355 | 0.304 | 1.000 |
| 9. Vegan vs. Vegetarian but eat seafood | 13 | 0.0451 | 0.519 | 0.974 | 1.000 |
| 10. Vegetarian vs. Vegetarian but eat seafood | 14 | 0.1006 | 1.343 | 0.193 | 1.000 |

None of the ten exploratory pairwise comparisons was statistically significant after Holm correction.

#### 4.1.2 16-mers

We repeat the ten unadjusted pairwise PERMANOVAs using 16-mer Bray–Curtis distances and the same samples for each diet pair. Diet is the only predictor.

In [53]:
for pair_number, (group_a, group_b) in enumerate(diet_pairs, start=1):
    pair_distance_path = f"{pairwise_input_dir}/pair_{pair_number}_kmer16.qza"
    pair_metadata_path = f"{pairwise_input_dir}/pair_{pair_number}_metadata.tsv"
    pair_qzv_path = f"{unadjusted_permanova_dir}/kmer16_pair_{pair_number:02d}.qzv"

    print(f"{pair_number}. {group_a} vs. {group_b}")

    !qiime diversity adonis \
        --i-distance-matrix "$pair_distance_path" \
        --m-metadata-file "$pair_metadata_path" \
        --p-formula "diet_type_sample" \
        --o-visualization "$pair_qzv_path"

1. Omnivore vs. Omnivore but do not eat red meat
Saved Visualization to: ../data/processed/09_beta_diversity_diet/unadjusted_permanova/kmer16_pair_01.qzv
2. Omnivore vs. Vegan
Saved Visualization to: ../data/processed/09_beta_diversity_diet/unadjusted_permanova/kmer16_pair_02.qzv
3. Omnivore vs. Vegetarian
Saved Visualization to: ../data/processed/09_beta_diversity_diet/unadjusted_permanova/kmer16_pair_03.qzv
4. Omnivore vs. Vegetarian but eat seafood
Saved Visualization to: ../data/processed/09_beta_diversity_diet/unadjusted_permanova/kmer16_pair_04.qzv
5. Omnivore but do not eat red meat vs. Vegan
Saved Visualization to: ../data/processed/09_beta_diversity_diet/unadjusted_permanova/kmer16_pair_05.qzv
6. Omnivore but do not eat red meat vs. Vegetarian
Saved Visualization to: ../data/processed/09_beta_diversity_diet/unadjusted_permanova/kmer16_pair_06.qzv
7. Omnivore but do not eat red meat vs. Vegetarian but eat seafood
Saved Visualization to: ../data/processed/09_beta_diversity_diet/

In [56]:
# Visualization.load(f"{unadjusted_permanova_dir}/kmer16_pair_01.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/kmer16_pair_02.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/kmer16_pair_03.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/kmer16_pair_04.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/kmer16_pair_05.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/kmer16_pair_06.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/kmer16_pair_07.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/kmer16_pair_08.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/kmer16_pair_09.qzv")
# Visualization.load(f"{unadjusted_permanova_dir}/kmer16_pair_10.qzv")

#### Unadjusted pairwise results · 16-mers

| Diet pair | Samples | R² | Pseudo-F | p-value | Holm-adjusted p |
|---|---:|---:|---:|---:|---:|
| 1. Omnivore vs. Omnivore but do not eat red meat | 120 | 0.0036 | 0.427 | 0.971 | 1.000 |
| 2. Omnivore vs. Vegan | 118 | 0.0072 | 0.840 | 0.553 | 1.000 |
| 3. Omnivore vs. Vegetarian | 119 | 0.0059 | 0.696 | 0.715 | 1.000 |
| 4. Omnivore vs. Vegetarian but eat seafood | 121 | 0.0044 | 0.526 | 0.911 | 1.000 |
| 5. Omnivore but do not eat red meat vs. Vegan | 12 | 0.0823 | 0.896 | 0.413 | 1.000 |
| 6. Omnivore but do not eat red meat vs. Vegetarian | 13 | 0.0246 | 0.277 | 0.875 | 1.000 |
| 7. Omnivore but do not eat red meat vs. Vegetarian but eat seafood | 15 | 0.0536 | 0.736 | 0.690 | 1.000 |
| 8. Vegan vs. Vegetarian | 11 | 0.1143 | 1.162 | 0.292 | 1.000 |
| 9. Vegan vs. Vegetarian but eat seafood | 13 | 0.0487 | 0.563 | 0.894 | 1.000 |
| 10. Vegetarian vs. Vegetarian but eat seafood | 14 | 0.0703 | 0.907 | 0.466 | 1.000 |

None of the ten exploratory pairwise comparisons was statistically significant after Holm correction.

#### Unadjusted results

| Diet pair | Whole-ASV R² | 16-mer R² | Whole-ASV p | 16-mer p |
|---|---:|---:|---:|---:|
| Vegan vs. Vegetarian (n = 11) | 0.1308 | 0.1143 | 0.304 | 0.292 |
| Vegetarian vs. Vegetarian but eat seafood (n = 14) | 0.1006 | 0.0703 | 0.193 | 0.466 |

Vegan vs. Vegetarian had the highest observed R² for both representations and the lowest raw p-value among the 16-mer tests. Vegetarian vs. Vegetarian but eat seafood had the lowest raw p-value among the whole-ASV tests and the largest difference in R² between representations (3.03 percentage points). R² was higher in both overall PERMANOVA models and in eight of ten unadjusted pairwise comparisons. All pairwise Holm-adjusted p-values were 1.000, so we interpret these R² differences descriptively.

### 4.2 Adjusted pairwise PERMANOVA

We reuse the ten diet pairs and sample subsets from section 4.1. For each pair, we test the diet term after accounting for age and settlement type. We apply Holm correction across the ten comparisons separately for whole ASVs and 16-mers.

#### 4.2.1 Whole ASVs

In [13]:
for pair_number, (group_a, group_b) in enumerate(diet_pairs, start=1):
    pair_distance_path = f"{pairwise_input_dir}/pair_{pair_number}_asv.qza"
    pair_metadata_path = f"{pairwise_input_dir}/pair_{pair_number}_metadata.tsv"
    pair_qzv_path = f"{adjusted_permanova_dir}/asv_pair_{pair_number:02d}.qzv"

    print(f"{pair_number}. {group_a} vs. {group_b}")

    !qiime diversity adonis \
        --i-distance-matrix "$pair_distance_path" \
        --m-metadata-file "$pair_metadata_path" \
        --p-formula "age_years_sample + Settlement_Type + diet_type_sample" \
        --o-visualization "$pair_qzv_path"

1. Omnivore vs. Omnivore but do not eat red meat
Saved Visualization to: ../data/processed/09_beta_diversity_diet/adjusted_permanova/asv_pair_01.qzv
2. Omnivore vs. Vegan
Saved Visualization to: ../data/processed/09_beta_diversity_diet/adjusted_permanova/asv_pair_02.qzv
3. Omnivore vs. Vegetarian
Saved Visualization to: ../data/processed/09_beta_diversity_diet/adjusted_permanova/asv_pair_03.qzv
4. Omnivore vs. Vegetarian but eat seafood
Saved Visualization to: ../data/processed/09_beta_diversity_diet/adjusted_permanova/asv_pair_04.qzv
5. Omnivore but do not eat red meat vs. Vegan
Saved Visualization to: ../data/processed/09_beta_diversity_diet/adjusted_permanova/asv_pair_05.qzv
6. Omnivore but do not eat red meat vs. Vegetarian
Saved Visualization to: ../data/processed/09_beta_diversity_diet/adjusted_permanova/asv_pair_06.qzv
7. Omnivore but do not eat red meat vs. Vegetarian but eat seafood
Saved Visualization to: ../data/processed/09_beta_diversity_diet/adjusted_permanova/asv_pair_07

#### Adjusted pairwise results · Whole ASVs

| Diet pair | Samples | R² | Pseudo-F | p-value | Holm-adjusted p |
|---|---:|---:|---:|---:|---:|
| 1. Omnivore vs. Omnivore but do not eat red meat | 120 | 0.0034 | 0.402 | 0.996 | 1.000 |
| 2. Omnivore vs. Vegan | 118 | 0.0064 | 0.753 | 0.719 | 1.000 |
| 3. Omnivore vs. Vegetarian | 119 | 0.0071 | 0.852 | 0.600 | 1.000 |
| 4. Omnivore vs. Vegetarian but eat seafood | 121 | 0.0067 | 0.810 | 0.644 | 1.000 |
| 5. Omnivore but do not eat red meat vs. Vegan | 12 | 0.0637 | 0.665 | 0.737 | 1.000 |
| 6. Omnivore but do not eat red meat vs. Vegetarian | 13 | 0.0444 | 0.474 | 0.772 | 1.000 |
| 7. Omnivore but do not eat red meat vs. Vegetarian but eat seafood | 15 | 0.0508 | 0.721 | 0.752 | 1.000 |
| 8. Vegan vs. Vegetarian | 11 | 0.1293 | 1.331 | 0.215 | 1.000 |
| 9. Vegan vs. Vegetarian but eat seafood | 13 | 0.0575 | 0.632 | 0.898 | 1.000 |
| 10. Vegetarian vs. Vegetarian but eat seafood | 14 | 0.0962 | 1.422 | 0.140 | 1.000 |

After accounting for age and settlement type, none of the ten diet comparisons was statistically significant after Holm correction. Vegan vs. Vegetarian had the highest observed diet R², and Vegetarian vs. Vegetarian but eat seafood had the lowest raw diet p-value.

#### 4.2.2 16-mers

We repeat the ten adjusted pairwise PERMANOVAs using 16-mer Bray–Curtis distances and the same sample subsets as in 4.2.1. We test the diet term after accounting for age and settlement type.

In [14]:
for pair_number, (group_a, group_b) in enumerate(diet_pairs, start=1):
    pair_distance_path = f"{pairwise_input_dir}/pair_{pair_number}_kmer16.qza"
    pair_metadata_path = f"{pairwise_input_dir}/pair_{pair_number}_metadata.tsv"
    pair_qzv_path = f"{adjusted_permanova_dir}/kmer16_pair_{pair_number:02d}.qzv"

    print(f"{pair_number}. {group_a} vs. {group_b}")

    !qiime diversity adonis \
        --i-distance-matrix "$pair_distance_path" \
        --m-metadata-file "$pair_metadata_path" \
        --p-formula "age_years_sample + Settlement_Type + diet_type_sample" \
        --o-visualization "$pair_qzv_path"

1. Omnivore vs. Omnivore but do not eat red meat
Saved Visualization to: ../data/processed/09_beta_diversity_diet/adjusted_permanova/kmer16_pair_01.qzv
2. Omnivore vs. Vegan
Saved Visualization to: ../data/processed/09_beta_diversity_diet/adjusted_permanova/kmer16_pair_02.qzv
3. Omnivore vs. Vegetarian
Saved Visualization to: ../data/processed/09_beta_diversity_diet/adjusted_permanova/kmer16_pair_03.qzv
4. Omnivore vs. Vegetarian but eat seafood
Saved Visualization to: ../data/processed/09_beta_diversity_diet/adjusted_permanova/kmer16_pair_04.qzv
5. Omnivore but do not eat red meat vs. Vegan
Saved Visualization to: ../data/processed/09_beta_diversity_diet/adjusted_permanova/kmer16_pair_05.qzv
6. Omnivore but do not eat red meat vs. Vegetarian
Saved Visualization to: ../data/processed/09_beta_diversity_diet/adjusted_permanova/kmer16_pair_06.qzv
7. Omnivore but do not eat red meat vs. Vegetarian but eat seafood
Saved Visualization to: ../data/processed/09_beta_diversity_diet/adjusted_per

#### Adjusted pairwise results · 16-mers

| Diet pair | Samples | R² | Pseudo-F | p-value | Holm-adjusted p |
|---|---:|---:|---:|---:|---:|
| 1. Omnivore vs. Omnivore but do not eat red meat | 120 | 0.0025 | 0.293 | 0.997 | 1.000 |
| 2. Omnivore vs. Vegan | 118 | 0.0068 | 0.811 | 0.583 | 1.000 |
| 3. Omnivore vs. Vegetarian | 119 | 0.0044 | 0.528 | 0.889 | 1.000 |
| 4. Omnivore vs. Vegetarian but eat seafood | 121 | 0.0049 | 0.598 | 0.823 | 1.000 |
| 5. Omnivore but do not eat red meat vs. Vegan | 12 | 0.0431 | 0.467 | 0.891 | 1.000 |
| 6. Omnivore but do not eat red meat vs. Vegetarian | 13 | 0.0344 | 0.366 | 0.792 | 1.000 |
| 7. Omnivore but do not eat red meat vs. Vegetarian but eat seafood | 15 | 0.0440 | 0.625 | 0.783 | 1.000 |
| 8. Vegan vs. Vegetarian | 11 | 0.1476 | 1.590 | 0.162 | 1.000 |
| 9. Vegan vs. Vegetarian but eat seafood | 13 | 0.0715 | 0.800 | 0.699 | 1.000 |
| 10. Vegetarian vs. Vegetarian but eat seafood | 14 | 0.0653 | 0.919 | 0.468 | 1.000 |

After accounting for age and settlement type, none of the ten diet comparisons was statistically significant after Holm correction. Vegan vs. Vegetarian had the highest observed diet R² (0.1476) and the lowest raw p-value (0.162).

#### Adjusted results

| Diet pair | Whole-ASV R² | 16-mer R² | Whole-ASV raw p | 16-mer raw p |
|---|---:|---:|---:|---:|
| Vegan vs. Vegetarian (n = 11) | 0.1293 | 0.1476 | 0.215 | 0.162 |
| Vegetarian vs. Vegetarian but eat seafood (n = 14) | 0.0962 | 0.0653 | 0.140 | 0.468 |

Vegan vs. Vegetarian had the highest observed diet R² for both representations. The lowest raw p-value came from a different pair for each representation, as shown in the table.

Whole-ASV R² was higher in seven of ten comparisons. All Holm-adjusted p-values were 1.000 for both representations, so these differences are descriptive.

### 4.3 Compare unadjusted and adjusted pairwise PERMANOVA

Both models used the same samples for each diet pair. Across all ten comparisons, every Holm-adjusted p-value was 1.000 for both whole ASVs and 16-mers. The R² values and raw p-values below are descriptive.

| Diet pair | Representation | Diet R² (unadjusted → adjusted) | Raw p (unadjusted → adjusted) |
|---|---|---:|---:|
| Vegan vs. Vegetarian (n = 11) | Whole ASVs | 0.1308 → 0.1293 | 0.304 → 0.215 |
| Vegan vs. Vegetarian (n = 11) | 16-mers | 0.1143 → 0.1476 | 0.292 → 0.162 |
| Vegetarian vs. Vegetarian but eat seafood (n = 14) | Whole ASVs | 0.1006 → 0.0962 | 0.193 → 0.140 |
| Vegetarian vs. Vegetarian but eat seafood (n = 14) | 16-mers | 0.0703 → 0.0653 | 0.466 → 0.468 |

Vegan vs. Vegetarian had the highest observed diet R² for both representations in both models. After accounting for age and settlement type, its R² stayed similar for whole ASVs and increased for 16-mers. Across all ten pairs, whole-ASV R² was higher than 16-mer R² in eight unadjusted and seven adjusted comparisons. Neither model identified a statistically significant difference between any diet pair.

## 5. Discussion of overall and pairwise PERMANOVA

The overall tests provided little evidence of an association between reported diet and fungal community composition. This conclusion was consistent across whole-ASV and 16-mer Bray–Curtis distances, with and without adjustment for age and settlement type. None of the pairwise comparisons was statistically significant after Holm correction.

The overall diet R² ranged from 1.65% to 2.31%. Some pairwise R² values were larger, particularly for Vegan vs. Vegetarian, which included only 11 samples. Each pairwise R² describes variation within its own two-group subset, so it answers a different question from the overall R² across all five diet groups.